In [1]:
#%matplotlib inline
import pickle
import numpy as np
import math
import pywt
from biosppy.signals import tools


def statistics(data): # The function for the 4 statistics
    avg = np.mean(data) # mean
    sd = np.std(data) # standard deviation
    maxm = max(data) # maximum
    minm = min(data) # minimum
    s_mean, s_med, _, _, s_max, _, s_var, s_std_dev, s_abs_dev, _, s_kurtois, s_skew = tools.signal_stats(data)
    return avg, sd, maxm, minm, s_med, s_max, s_var, s_abs_dev, s_kurtois, s_skew
    
def Derivatives(data): # Get the first and second derivatives of the data
    deriv = (data[1:-1] + data[2:])/ 2. - (data[1:-1] + data[:-2])/ 2.
    secondDeriv = data[2:] - 2*data[1:-1] + data[:-2]
    return deriv,secondDeriv

def strided_app(a, L, S ):  # Window len = L, Stride len/stepsize = S
    nrows = ((a.size-L)//S)+1
    n = a.strides[0]
    return np.lib.stride_tricks.as_strided(a, shape=(nrows,L), strides=(S*n,n))


data_set = 'WESAD/'

s2_path = data_set + 'S2/S2.pkl'
s3_path = data_set + 'S3/S3.pkl'
s4_path = data_set + 'S4/S4.pkl'
s5_path = data_set + 'S5/S5.pkl'
s6_path = data_set + 'S6/S6.pkl'
s7_path = data_set + 'S7/S7.pkl'
s8_path = data_set + 'S8/S8.pkl'
s9_path = data_set + 'S9/S9.pkl'
s10_path = data_set + 'S10/S10.pkl'
s11_path = data_set + 'S11/S11.pkl'
s13_path = data_set + 'S13/S13.pkl'
s14_path = data_set + 'S14/S14.pkl'
s15_path = data_set + 'S15/S15.pkl'
s16_path = data_set + 'S16/S16.pkl'
s17_path = data_set + 'S17/S17.pkl'

all_path = [s2_path, s3_path,s4_path,s5_path,s6_path,s7_path,s8_path,s9_path,s10_path,s11_path,s13_path,s14_path,s15_path,s16_path,s17_path]
all_features = []
all_label = []
for subject_path in all_path:
    print(subject_path)
    with open(subject_path, 'rb') as file:
        s2_data = pickle.load(file, encoding='latin1')
        
    label_1 = np.where(s2_data['label'] == 1)
    start_1 = math.ceil(label_1[0][0]/175)
    end_1 = math.floor(label_1[0][len(label_1[0])-1] / 175)

    label_2 = np.where(s2_data['label'] == 2)
    start_2 = math.ceil(label_2[0][0]/175)
    end_2 = math.floor(label_2[0][len(label_2[0])-1] / 175)

    label_3 = np.where(s2_data['label'] == 3)
    start_3 = math.ceil(label_3[0][0]/175)
    end_3 = math.floor(label_3[0][len(label_3[0])-1] / 175)

    eda_data = s2_data['signal']['wrist']['EDA'][:,0]
    acc_data = s2_data['signal']['wrist']['ACC']
    bvp_data = s2_data['signal']['wrist']['BVP'][:,0]
    temp_data = s2_data['signal']['wrist']['TEMP'][:,0]

    eda_data_1 = eda_data[start_1:end_1+1]
    eda_data_2 = eda_data[start_2:end_2+1]
    eda_data_3 = eda_data[start_3:end_3+1]

    acc_data_1 = acc_data[start_1*8:end_1*8+8,:]
    acc_data_2 = acc_data[start_2*8:end_2*8+8,:]
    acc_data_3 = acc_data[start_3*8:end_3*8+8,:]

    bvp_data_1 = bvp_data[start_1*16:end_1*16+16]
    bvp_data_2 = bvp_data[start_2*16:end_2*16+16]
    bvp_data_3 = bvp_data[start_3*16:end_3*16+16]

    temp_data_1 = temp_data[start_1*1:end_1*1+1]
    temp_data_2 = temp_data[start_2*1:end_2*1+1]
    temp_data_3 = temp_data[start_3*1:end_3*1+1]

    EDA_1 = strided_app(eda_data_1, 240, 1) # Window len = L, Stride len/stepsize = S
    EDA_2 = strided_app(eda_data_2, 240, 1)
    EDA_3 = strided_app(eda_data_3, 240, 1)

    ACCx_1 = strided_app(acc_data_1[:,0], 1920, 8) # Window len = L, Stride len/stepsize = S
    ACCy_1 = strided_app(acc_data_1[:,1], 1920, 8)
    ACCz_1 = strided_app(acc_data_1[:,2], 1920, 8)
    ACCx_2 = strided_app(acc_data_2[:,0], 1920, 8)
    ACCy_2 = strided_app(acc_data_2[:,1], 1920, 8)
    ACCz_2 = strided_app(acc_data_2[:,2], 1920, 8)
    ACCx_3 = strided_app(acc_data_3[:,0], 1920, 8)
    ACCy_3 = strided_app(acc_data_3[:,1], 1920, 8)
    ACCz_3 = strided_app(acc_data_3[:,2], 1920, 8)

    BVP_1 = strided_app(bvp_data_1, 3840, 16)
    BVP_2 = strided_app(bvp_data_2, 3840, 16)
    BVP_3 = strided_app(bvp_data_3, 3840, 16)

    TEMP_1 = strided_app(temp_data_1, 240, 1)
    TEMP_2 = strided_app(temp_data_2, 240, 1)
    TEMP_3 = strided_app(temp_data_3, 240, 1)

    EDA = np.concatenate((EDA_1, EDA_2, EDA_3),axis=0) 
    ACCx = np.concatenate((ACCx_1, ACCx_2, ACCx_3),axis=0) 
    ACCy = np.concatenate((ACCy_1, ACCy_2, ACCy_3),axis=0) 
    ACCz = np.concatenate((ACCz_1, ACCz_2, ACCz_3),axis=0)
    
    ACC = np.sqrt(np.square(ACCx) + np.square(ACCy) + np.square(ACCz))
    
    BVP = np.concatenate((BVP_1, BVP_2, BVP_3),axis=0) 

    smooth_signal = tools.smoother(BVP, kernel='median', size=5) # Convolutional 5x5 kernel window
    data = smooth_signal['signal']

    norm_signal = tools.normalize(data)
    BVP = norm_signal['signal']

    TEMP = np.concatenate((TEMP_1, TEMP_2, TEMP_3),axis=0) 

    label_1 = [1] * len(EDA_1)
    label_2 = [2] * len(EDA_2)
    label_3 = [1] * len(EDA_3)

    LABEL = np.concatenate((label_1, label_2, label_3),axis=0) 
    
    all_label.append(LABEL)

    # Construct the feature matrix, 60 EDA features, 240 ACC features, 60 TEMP features, 60 BVA features, and 420 features in total. 
    
    length=len(LABEL)
    features = np.zeros((length,420))

    for i in range(length):
        if(i%500==0):
            print(i)
        deriv_EDA,secondDeriv_EDA = Derivatives(EDA[i,:])
        deriv_ACC,secondDeriv_ACC = Derivatives(ACC[i,:])
        deriv_ACCx,secondDeriv_ACCx = Derivatives(ACCx[i,:])
        deriv_ACCy,secondDeriv_ACCy = Derivatives(ACCy[i,:])
        deriv_ACCz,secondDeriv_ACCz = Derivatives(ACCz[i,:])
        _, EDA_cD_3, EDA_cD_2, EDA_cD_1 = pywt.wavedec(EDA[i,:], 'Haar', level=3) #3 = 1Hz, 2 = 2Hz, 1=4Hz
        _, ACC_cD_3, ACC_cD_2, ACC_cD_1 = pywt.wavedec(ACC[i,:], 'Haar', level=3) 
        _, ACCx_cD_3, ACCx_cD_2, ACCx_cD_1 = pywt.wavedec(ACCx[i,:], 'Haar', level=3) 
        _, ACCy_cD_3, ACCy_cD_2, ACCy_cD_1 = pywt.wavedec(ACCy[i,:], 'Haar', level=3) 
        _, ACCz_cD_3, ACCz_cD_2, ACCz_cD_1 = pywt.wavedec(ACCz[i,:], 'Haar', level=3) 

        deriv_TEMP,secondDeriv_TEMP = Derivatives(TEMP[i,:])
        _, TEMP_cD_3, TEMP_cD_2, TEMP_cD_1 = pywt.wavedec(TEMP[i,:], 'Haar', level=3)
        
        deriv_BVP,secondDeriv_BVP = Derivatives(BVP[i,:])
        _, BVP_cD_3, BVP_cD_2, BVP_cD_1 = pywt.wavedec(BVP[i,:], 'Haar', level=3)
        

        ### EDA features
        # EDA statistical features:
        features[i,0:10] = statistics(EDA[i,:])
        features[i,10:20] = statistics(deriv_EDA)
        features[i,20:30] = statistics(secondDeriv_EDA)
        # EDA wavelet features:
        features[i,30:40] = statistics(EDA_cD_3)
        features[i,40:50] = statistics(EDA_cD_2)
        features[i,50:60] = statistics(EDA_cD_1)

        ### ACC features
        ## ACC statistical features:
        # Acceleration magnitude:
        features[i,60:70] = statistics(ACC[i,:])
        features[i,70:80] = statistics(deriv_ACC)
        features[i,80:90] = statistics(secondDeriv_ACC)
        # Acceleration x-axis:
        features[i,90:100] = statistics(ACCx[i,:])
        features[i,100:110] = statistics(deriv_ACCx)
        features[i,110:120] = statistics(secondDeriv_ACCx)
        # Acceleration y-axis:
        features[i,120:130] = statistics(ACCy[i,:])
        features[i,130:140] = statistics(deriv_ACCy)
        features[i,140:150] = statistics(secondDeriv_ACCy)
        # Acceleration z-axis:
        features[i,150:160] = statistics(ACCz[i,:])
        features[i,160:170] = statistics(deriv_ACCz)
        features[i,170:180] = statistics(secondDeriv_ACCz)
        ## ACC wavelet features:
        # ACC magnitude wavelet features:
        features[i,180:190] = statistics(ACC_cD_3)
        features[i,190:200] = statistics(ACC_cD_2)
        features[i,200:210] = statistics(ACC_cD_1)
        # ACC x-axis wavelet features:
        features[i,210:220] = statistics(ACCx_cD_3)
        features[i,220:230] = statistics(ACCx_cD_2)
        features[i,230:240] = statistics(ACCx_cD_1)
        # ACC y-axis wavelet features:
        features[i,240:250] = statistics(ACCy_cD_3)
        features[i,250:260] = statistics(ACCy_cD_2)
        features[i,260:270] = statistics(ACCy_cD_1)
        # ACC z-axis wavelet features:
        features[i,270:280] = statistics(ACCz_cD_3)
        features[i,280:290] = statistics(ACCz_cD_2)
        features[i,290:300] = statistics(ACCz_cD_1)

        ### TEMP features
        # TEMP statistical features:
        features[i,300:310] = statistics(TEMP[i,:])
        features[i,310:320] = statistics(deriv_TEMP)
        features[i,320:330] = statistics(secondDeriv_TEMP)
        # TEMP wavelet features:
        features[i,330:340] = statistics(TEMP_cD_3)
        features[i,340:350] = statistics(TEMP_cD_2)
        features[i,350:360] = statistics(TEMP_cD_1)
        
        ### BVP features
        # BVP statistical features:
        features[i,360:370] = statistics(BVP[i,:])
        features[i,370:380] = statistics(deriv_BVP)
        features[i,380:390] = statistics(secondDeriv_BVP)
        # BVP wavelet features:
        features[i,390:400] = statistics(BVP_cD_3)
        features[i,400:410] = statistics(BVP_cD_2)
        features[i,410:420] = statistics(BVP_cD_1)

        
    all_features.append(features)

file_to_store = open("all_featuresv2.pickle", "wb")
pickle.dump(all_features, file_to_store)
file_to_store.close()

file_to_store = open("all_labelv2.pickle", "wb")
pickle.dump(all_label, file_to_store)
file_to_store.close()

WESAD/S2/S2.pkl
0
500
1000
1500
2000
2500
3000
3500
4000
4500
5000
5500
6000
6500
7000
7500
WESAD/S3/S3.pkl
0
500
1000
1500
2000
2500
3000
3500
4000
4500
5000
5500
6000
6500
7000
7500
WESAD/S4/S4.pkl
0
500
1000
1500
2000
2500
3000
3500
4000
4500
5000
5500
6000
6500
7000
7500
WESAD/S5/S5.pkl
0
500
1000
1500
2000
2500
3000
3500


/Users/cedric/miniconda3/envs/wesad/lib/python3.11/site-packages/biosppy/signals/tools.py:1028: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  kurt = stats.kurtosis(signal, bias=False)
/Users/cedric/miniconda3/envs/wesad/lib/python3.11/site-packages/biosppy/signals/tools.py:1031: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  skew = stats.skew(signal, bias=False)


4000
4500
5000
5500
6000
6500
7000
7500
8000
WESAD/S6/S6.pkl
0
500
1000
1500
2000
2500
3000
3500
4000
4500
5000
5500
6000
6500
7000
7500
8000
WESAD/S7/S7.pkl
0
500
1000
1500
2000
2500
3000
3500
4000
4500
5000
5500
6000
6500
7000
7500
8000
WESAD/S8/S8.pkl
0
500
1000
1500
2000
2500
3000
3500
4000
4500
5000
5500
6000
6500
7000
7500
8000
WESAD/S9/S9.pkl
0
500
1000
1500
2000
2500
3000
3500
4000
4500
5000
5500
6000
6500
7000
7500
8000
WESAD/S10/S10.pkl
0
500
1000
1500
2000
2500
3000
3500
4000
4500
5000
5500
6000
6500
7000
7500
8000
WESAD/S11/S11.pkl
0
500
1000
1500
2000
2500
3000
3500
4000
4500
5000
5500
6000
6500
7000
7500
8000
WESAD/S13/S13.pkl
0
500
1000
1500
2000
2500
3000
3500
4000
4500
5000
5500
6000
6500
7000
7500
8000
WESAD/S14/S14.pkl
0
500
1000
1500
2000
2500
3000
3500
4000
4500
5000
5500
6000
6500
7000
7500
8000
WESAD/S15/S15.pkl
0
500
1000
1500
2000
2500
3000
3500
4000
4500
5000
5500
6000
6500
7000
7500
8000
WESAD/S16/S16.pkl
0
500
1000
1500
2000
2500
3000
3500
4000
4500
5000
550